<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/SayisalGoruntuIslemeFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Sayın Hocam,**

Ödevi incelerken aşağıdaki hususları dikkate almanızı rica ederim. Kodlamaları ve işlemlerin detaylarını araştırdım; ancak yorumlama ve karşılaştırma açısından yeterli bilgiye sahip olmadığım için yorumlamaları sınırlı tutmak durumunda kaldım.

Kaggle veri indirme ve yükleme süreçlerinde, veri boyutunun yüksek olması nedeniyle doğrudan klasör olarak indirme yapılamamıştır. Bu nedenle veri, Kaggle API kullanılarak .json formatında indirilmiş ve Colab ortamına yüklenmiştir. Colab üzerinde veri erişimi için ilgili kaggle.json dosyasının yolu tanımlanarak API yetkilendirmesi gerçekleştirilmiştir. **Uygulamanın çalıştırılabilmesi için önceden KAGGLE_USERNAME ve KAGGLE_KEY değişkenlerinin tanımlanması gerekmektedir.**

Bilgilerinize sunarım.

Saygılarımla

In [ ]:
from google.colab import userdata
import os
import zipfile

In [ ]:

# Gizli anahtarları ilgili ortam değişkenlerine atama

print("Colab Secrets'tan Kaggle API bilgileri okunuyor...")
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("API bilgileri başarıyla okundu.")
except userdata.SecretNotFoundError:
    print("\nHata: 'KAGGLE_USERNAME' veya 'KAGGLE_KEY' gizli anahtarları bulunamadı.")
    print("Lütfen anahtar oluşturun")

In [ ]:
!pip install -q kaggle

In [ ]:
!kaggle datasets download -d nodoubttome/skin-cancer9-classesisic

In [ ]:

zip_file_path = '/content/skin-cancer9-classesisic.zip'
extract_dir = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")

print("Çıkarılan dizinin içeriği:")
for item in os.listdir(extract_dir):
    print(item)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
import cv2
from PIL import Image
from collections import Counter
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Veri Setinin Yüklenmesi

data_path = '/content/Skin cancer ISIC The International Skin Imaging Collaboration/Train'

In [ ]:
image_paths = []
labels = []

for class_name in os.listdir(data_path):
    class_path = os.path.join(data_path, class_name)

    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):

            if img_name.endswith(('.jpg', '.png', '.jpeg')):
                image_paths.append(os.path.join(class_path, img_name))
                labels.append(class_name)

print(f"Toplam {len(image_paths)} görüntü yolu ve etiket toplandı.")

In [ ]:
def analyze_image_features(df):


    resolutions = []
    channels = []
    file_sizes = []

    print("Görüntü özellikleri analiz ediliyor...")

    for index, row in df.iterrows():
        image_path = None
        try:
            image_path = row['image_path']
            img = Image.open(image_path)
            resolutions.append(img.size)


            if img.mode == 'RGB' or img.mode == 'RGBA':
                channels.append(len(img.getbands()))
            elif img.mode == 'L':
                channels.append(1)
            else:

                channels.append(len(img.getbands()))


            file_sizes.append(os.path.getsize(image_path))

            img.close()

        except Exception as e:

            print(f"Hata oluştu: {image_path} - {e}")
            resolutions.append(None)
            channels.append(None)
            file_sizes.append(None)


    df['cozunurluk'] = resolutions
    df['kanal_sayisi'] = channels
    df['dosya_boyutu_byte'] = file_sizes

    return df

In [ ]:
#Veri Özelliklerinin İncelenmesi
data = {'image_path': image_paths, 'label': labels}
train_df = pd.DataFrame(data)

train_df = analyze_image_features(train_df)

print("\n--- Çözünürlük Dağılımı ---")

cozunurluk_dagilimi = train_df['cozunurluk'].value_counts()
print(cozunurluk_dagilimi.head(10))

print("\n--- Kanal Sayısı Dağılımı ---")
train_df['genislik'] = train_df['cozunurluk'].apply(lambda x: x[0] if x is not None else None)
train_df['yukseklik'] = train_df['cozunurluk'].apply(lambda x: x[1] if x is not None else None)

print("\n--- Genişlik ve Yükseklik İstatistikleri ---")
print(train_df[['genislik', 'yukseklik']].describe())

**Görüntü Yükleme ve Görselleştirme - (1) RGB → Grayscale Dönüşümü )**

In [ ]:

def visualize_random_images(df, n_images=20):
    """
    DataFrame'den rastgele n_images adet görüntü seçer ve her birinin
    RGB ve Grayscale versiyonlarını yan yana gösterir.
    """


    if len(df) < n_images:
        print(f"Hata: DataFrame'de istenen {n_images} adetten daha az ({len(df)}) görüntü var.")
        return

    random_samples = df.sample(n=n_images, random_state=42)

    fig, axes = plt.subplots(n_images, 2, figsize=(10, 5 * n_images))
    plt.tight_layout(pad=3.0)

    # 9'dan az görüntü olması ihtimaline karşı axes'in boyutunu kontrol et
    if n_images == 1:
        axes = np.array([axes])

    print(f"Rastgele seçilen {n_images} görüntü görselleştiriliyor...")

    for i, (index, row) in enumerate(random_samples.iterrows()):
        image_path = row['image_path'] # 'dosya_yolu' yerine 'image_path' kullanıldı

        try:
            # 1. Görüntüyü yükle (OpenCV varsayılan olarak BGR formatında yükler)
            img_bgr = cv2.imread(image_path)

            # 2. RGB'ye dönüştür (Matplotlib için gereklidir)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            # 3. Grayscale'e dönüştür
            img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

            # --- Görselleştirme ---

            # Sol Sütun: RGB Hali
            axes[i, 0].imshow(img_rgb)
            axes[i, 0].set_title(f"({i+1}) RGB - Etiket: {row.get('label', 'Yok')}", fontsize=10) # 'etiket' yerine 'label' kullanıldı
            axes[i, 0].axis('off')

            # Sağ Sütun: Grayscale Hali
            axes[i, 1].imshow(img_gray, cmap='gray') # Grayscale için cmap='gray' şarttır
            axes[i, 1].set_title(f"({i+1}) Grayscale", fontsize=10)
            axes[i, 1].axis('off')

        except Exception as e:
            # Okuma hatası durumunda yer tutucu
            axes[i, 0].set_title(f"Hata: {image_path}", fontsize=10)
            axes[i, 0].axis('off')
            axes[i, 1].axis('off')
            print(f"Görüntü okuma hatası: {image_path} - {e}")

    plt.show()

In [ ]:

def calculate_image_statistics(df, n_samples=20):
    """
    DataFrame'den rastgele n_samples adet görüntü seçer ve her birinin
    RGB ve Grayscale formatlarında istatistiksel özelliklerini hesaplar.
    """

    # DataFrame'den rastgele n_samples adet satır seçme
    if len(df) < n_samples:
        print(f"Hata: DataFrame'de istenen {n_samples} adetten daha az ({len(df)}) görüntü var.")
        return pd.DataFrame() # Boş DataFrame döndür

    random_samples = df.sample(n=n_samples, random_state=42)

    results = []

    print(f"Rastgele seçilen {n_samples} görüntünün istatistikleri hesaplanıyor...")

    for i, (index, row) in enumerate(random_samples.iterrows()):
        image_path = row['image_path'] # Changed from 'dosya_yolu' to 'image_path'

        try:
            # Görüntüyü yükle (BGR formatında)
            img_bgr = cv2.imread(image_path)

            # --- RGB İstatistikleri ---
            # BGR'yi RGB'ye dönüştür
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            # Düzleştirilmiş piksel değerleri (tek boyutlu dizi)
            rgb_pixels = img_rgb.flatten()

            rgb_min = np.min(rgb_pixels)
            rgb_max = np.max(rgb_pixels)
            rgb_mean = np.mean(rgb_pixels)
            rgb_std = np.std(rgb_pixels)

            # --- Grayscale İstatistikleri ---
            img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
            gray_pixels = img_gray.flatten()

            gray_min = np.min(gray_pixels)
            gray_max = np.max(gray_pixels)
            gray_mean = np.mean(gray_pixels)
            gray_std = np.std(gray_pixels)

            # Sonuçları listeye ekle
            results.append({
                'ID': i + 1,
                'Dosya Adı': os.path.basename(image_path),
                'Format': 'RGB',
                'Min': rgb_min,
                'Max': rgb_max,
                'Ortalama': f"{rgb_mean:.2f}",
                'Std. Sapma': f"{rgb_std:.2f}"
            })

            results.append({
                'ID': i + 1,
                'Dosya Adı': os.path.basename(image_path),
                'Format': 'Grayscale',
                'Min': gray_min,
                'Max': gray_max,
                'Ortalama': f"{gray_mean:.2f}",
                'Std. Sapma': f"{gray_std:.2f}"
            })

        except Exception as e:
            print(f"İstatistik hesaplama hatası: {image_path} - {e}")
            results.append({'ID': i + 1, 'Dosya Adı': os.path.basename(image_path), 'Format': 'HATA', 'Min': 'N/A', 'Max': 'N/A', 'Ortalama': 'N/A', 'Std. Sapma': 'N/A'})

    return pd.DataFrame(results)



In [ ]:
visualize_random_images(train_df, n_images=20)

** Segmentasyon Hattı**

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

if 'train_df' not in locals():

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

random_indices = np.random.choice(train_df.index, size=1, replace=False)
sample_idx = random_indices[0]
img_path = train_df.iloc[sample_idx]['image_path']

img_rgb_original = cv2.imread(img_path)
img_rgb_original = cv2.cvtColor(img_rgb_original, cv2.COLOR_BGR2RGB)
img_rgb_original = cv2.resize(img_rgb_original, (224, 224))

img_gray_original = cv2.cvtColor(img_rgb_original, cv2.COLOR_RGB2GRAY)

print(f"Segmentasyon için örnek görüntü seçildi: {os.path.basename(img_path)}")

# 1. Ön İşleme: Gürültüyü azaltmak için Gaussian Blur uygulama
img_gray_blurred = cv2.GaussianBlur(img_gray_original, (5, 5), 0)

# 2. Eşikleme: Otsu's thresholding ile ikili maske oluşturma
_, binary_mask = cv2.threshold(img_gray_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Morfolojik İşlemler: Maskeyi iyileştirmek için 'Açma' ve 'Kapama' uygulama
kernel = np.ones((5, 5), np.uint8)

# Açma: Küçük beyaz gürültüleri (ön plan) kaldırır
mask_opened = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel, iterations=2)

# Kapama: Küçük siyah delikleri (arka plan) doldurur
mask_closed = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel, iterations=2)

# 4. Kontur Bulma (İsteğe bağlı görselleştirme için)
contours, _ = cv2.findContours(mask_closed.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Orijinal RGB görüntü üzerinde konturları çizme
img_rgb_contours = img_rgb_original.copy()
cv2.drawContours(img_rgb_contours, contours, -1, (0, 255, 0), 2)

# Maskeyi orijinal RGB görüntünün üzerine bindirme
mask_overlay = cv2.cvtColor(mask_closed, cv2.COLOR_GRAY2BGR) / 255.0

# Orijinal RGB görüntüyü (0,1) aralığına normalize edelim
img_rgb_norm = img_rgb_original / 255.0

# Bindirme işlemi (alpha blending)
alpha = 0.4 # Maskenin şeffaflık derecesi
segmented_overlay = (img_rgb_norm * (1 - alpha)) + (mask_overlay * alpha * np.array([0,1,0])) # Yeşil maske bindirmesi
segmented_overlay = np.clip(segmented_overlay * 255, 0, 255).astype(np.uint8)

# Görselleştirme
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Grayscale Görüntü Segmentasyon Hattı', fontsize=16)

axes[0].imshow(img_gray_original, cmap='gray')
axes[0].set_title('Orijinal Grayscale')
axes[0].axis('off')

axes[1].imshow(binary_mask, cmap='gray')
axes[1].set_title('Otsu Eşikleme Sonucu')
axes[1].axis('off')

axes[2].imshow(mask_closed, cmap='gray')
axes[2].set_title('Morfolojik İşlemler Sonucu (Maske)')
axes[2].axis('off')

axes[3].imshow(segmented_overlay)
axes[3].set_title('Segmentasyon Maskesi (RGB Üzerinde)')
axes[3].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**Görüntü İşleme ve İyileştirme - Pre-Processing**

In [ ]:
def contrast_stretching(img):
    """Kontrast germe işlemi"""
    if len(img.shape) == 3:  # RGB
        result = np.zeros_like(img)
        for i in range(3):
            channel = img[:, :, i]
            min_val = channel.min()
            max_val = channel.max()
            result[:, :, i] = ((channel - min_val) / (max_val - min_val) * 255).astype(np.uint8)
        return result
    else:  # Grayscale
        min_val = img.min()
        max_val = img.max()
        return ((img - min_val) / (max_val - min_val) * 255).astype(np.uint8)

In [ ]:
visualize_random_images(train_df, n_images=9)

print("Yukarıdaki görseller, rastgele seçilen 9 görüntünün orijinal RGB hallerini ve gri tonlamalı (grayscale) versiyonlarını yan yana göstermektedir.")
print("Bu dönüşüm, görüntülerin renk bilgilerini kaybederek sadece parlaklık değerlerini korumasını sağlar ve segmentasyon veya özellik çıkarımı gibi sonraki işlem adımları için önemlidir.")

**Morfolojik İşlemler (Erozyon, Genişleme, Açma, Kapama)**


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

if 'img_rgb' not in locals() or 'img_gray' not in locals():
    if 'train_df' not in locals():
        data = {'image_path': image_paths, 'label': labels}
        train_df = pd.DataFrame(data)

    random_indices = np.random.choice(train_df.index, size=1, replace=False)
    sample_idx = random_indices[0]
    img_path = train_df.iloc[sample_idx]['image_path']
    img_rgb = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
    img_rgb = cv2.resize(img_rgb, (224, 224))
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)


# 1. (5,5) boyutunda elips şeklinde bir kernel oluşturma
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

# 2. Erozyon işlemi
img_rgb_eroded = cv2.erode(img_rgb, kernel, iterations=1)
img_gray_eroded = cv2.erode(img_gray, kernel, iterations=1)

# 3. Genişleme (Dilation) işlemi
img_rgb_dilated = cv2.dilate(img_rgb, kernel, iterations=1)
img_gray_dilated = cv2.dilate(img_gray, kernel, iterations=1)

# 4. Açma (Opening) işlemi
img_rgb_opened = cv2.morphologyEx(img_rgb, cv2.MORPH_OPEN, kernel)
img_gray_opened = cv2.morphologyEx(img_gray, cv2.MORPH_OPEN, kernel)

# 5. Kapama (Closing) işlemi
img_rgb_closed = cv2.morphologyEx(img_rgb, cv2.MORPH_CLOSE, kernel)
img_gray_closed = cv2.morphologyEx(img_gray, cv2.MORPH_CLOSE, kernel)

print("Morfolojik işlemler başarıyla uygulandı.")

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 20))
fig.suptitle('Morfolojik İşlemler Karşılaştırması', fontsize=16)

# Orijinal Görüntüler
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title('Orijinal RGB')
axes[0, 0].axis('off')
axes[0, 1].imshow(img_gray, cmap='gray')
axes[0, 1].set_title('Orijinal Grayscale')
axes[0, 1].axis('off')
axes[0, 2].axis('off') # Boş bırak

# Erozyon
axes[1, 0].imshow(img_rgb_eroded)
axes[1, 0].set_title('Erozyon RGB')
axes[1, 0].axis('off')
axes[1, 1].imshow(img_gray_eroded, cmap='gray')
axes[1, 1].set_title('Erozyon Grayscale')
axes[1, 1].axis('off')
axes[1, 2].text(0.5, 0.5, 'Erozyon, beyaz gürültüyü veya küçük nesneleri kaldırarak\n nesnelerin boyutunu küçültür.', fontsize=10, ha='center', va='center')
axes[1, 2].axis('off')

# Genişleme
axes[2, 0].imshow(img_rgb_dilated)
axes[2, 0].set_title('Genişleme RGB')
axes[2, 0].axis('off')
axes[2, 1].imshow(img_gray_dilated, cmap='gray')
axes[2, 1].set_title('Genişleme Grayscale')
axes[2, 1].axis('off')
axes[2, 2].text(0.5, 0.5, 'Genişleme, nesnelerin boyutunu büyüterek\n karanlık bölgelerdeki küçük delikleri doldurur.', fontsize=10, ha='center', va='center')
axes[2, 2].axis('off')

# Açma ve Kapama
axes[3, 0].imshow(img_rgb_opened)
axes[3, 0].set_title('Açma RGB')
axes[3, 0].axis('off')
axes[3, 1].imshow(img_gray_opened, cmap='gray')
axes[3, 1].set_title('Açma Grayscale')
axes[3, 1].axis('off')
axes[3, 2].text(0.5, 0.6, 'Açma (Erozyon sonra Genişleme): Küçük nesneleri veya\n çıkıntıları kaldırır.', fontsize=10, ha='center', va='center')

axes[3, 2].text(0.5, 0.3, 'Kapama (Genişleme sonra Erozyon): Küçük delikleri veya\n boşlukları doldurur.', fontsize=10, ha='center', va='center')
axes[3, 2].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Morfolojik işlemler ve görselleştirmeler tamamlandı.")

## Görüntü Kırpma Fonksiyonu Tanımlama




In [ ]:
def crop_image(img, strategy='fixed_percentage'):
    """
    Görüntüyü belirlenen stratejiye göre kenarlarından kırpar.
    Lezyon gibi ilgi alanlarını korurken gürültülü çerçeve bölgelerini kaldırır.

    Args:
        img (numpy.ndarray): Kırpılacak görüntü (OpenCV formatında).
        strategy (str): Kırpma stratejisi. 'fixed_percentage' veya 'dynamic' olabilir.

    Returns:
        numpy.ndarray: Kırpılmış görüntü.
    """

    if strategy == 'fixed_percentage':
        # Görüntünün %5'ini kenarlardan kırp
        padding_percentage = 0.05
        h, w = img.shape[:2]

        dx = int(w * padding_percentage)
        dy = int(h * padding_percentage)

        start_x = dx
        end_x = w - dx
        start_y = dy
        end_y = h - dy

        # Kırpma alanının geçerli olduğundan emin ol
        if start_x >= end_x or start_y >= end_y:
            print("Uyarı: Fixed percentage kırpma, görüntüyü çok küçültüyor veya geçersiz kılıyor. Orijinal görüntü döndürülüyor.")
            return img

        cropped_img = img[start_y:end_y, start_x:end_x]

    elif strategy == 'dynamic':
        # Dinamik kırpma için gri tonlamalıya dönüştür
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

        # Otsu's eşikleme ile ikili maske oluştur
        _, binary_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Kontur bul
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print("Uyarı: Dinamik kırpma için kontur bulunamadı. Orijinal görüntü döndürülüyor.")
            return img

        # En büyük alanı olan konturu seç
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)

        # Sınırlayıcı dikdörtgene küçük bir dolgu ekle
        padding = 15 # Piksel cinsinden dolgu miktarı
        h_img, w_img = img.shape[:2]

        start_x = max(0, x - padding)
        start_y = max(0, y - padding)
        end_x = min(w_img, x + w + padding)
        end_y = min(h_img, y + h + padding)

        # Kırpma alanının geçerli olduğundan emin ol
        if start_x >= end_x or start_y >= end_y:
            print("Uyarı: Dinamik kırpma, görüntüyü çok küçültüyor veya geçersiz kılıyor. Orijinal görüntü döndürülüyor.")
            return img

        cropped_img = img[start_y:end_y, start_x:end_x]

    else:
        print(f"Hata: Geçersiz kırpma stratejisi '{strategy}'. 'fixed_percentage' veya 'dynamic' olmalı.")
        return img

    return cropped_img

print("crop_image fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

if 'train_df' not in locals():
    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

# 1. train_df DataFrame'inden rastgele 9 adet görüntü seçin.
num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 3, figsize=(15, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} görüntü üzerinde kırpma işlemleri görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        # 2a. Görüntüyü yükleyin ve BGR'den RGB'ye dönüştürün.
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 2b. 'fixed_percentage' stratejisiyle kırpma.
        cropped_fixed = crop_image(img_rgb.copy(), strategy='fixed_percentage')

        # 2c. 'dynamic' stratejisiyle kırpma.
        cropped_dynamic = crop_image(img_rgb.copy(), strategy='dynamic')

        # 4. ve 5. Görselleştirme
        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"Orijinal - Etiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cropped_fixed)
        axes[i, 1].set_title('Sabit Yüzde Kırpılmış', fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(cropped_dynamic)
        axes[i, 2].set_title('Dinamik Kırpılmış', fontsize=10)
        axes[i, 2].axis('off')

    except Exception as e:
        print(f"Görüntü işleme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        axes[i, 0].axis('off')
        axes[i, 1].axis('off')
        axes[i, 2].axis('off')

# 6. Çizilen tüm grafikleri görüntüleyin.
plt.show()

## Kontrast İyileştirme Fonksiyonu Tanımlama



In [ ]:
def histogram_equalization(img):
    """Histogram eşitleme işlemi"""
    if len(img.shape) == 3:

        img_ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
        img_ycrcb[:, :, 0] = cv2.equalizeHist(img_ycrcb[:, :, 0])
        return cv2.cvtColor(img_ycrcb, cv2.COLOR_YCrCb2RGB)
    else:
        return cv2.equalizeHist(img)

print("histogram_equalization fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

if 'train_df' not in locals():
    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

# 1. train_df DataFrame'inden rastgele 9 adet görüntü seçin.
num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 4, figsize=(20, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} kırpılmış görüntü üzerinde kontrast iyileştirme işlemleri görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        # 2a. Görüntüyü yükleyin ve RGB formatına dönüştürün.
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 2b. `crop_image` fonksiyonunu kullanarak görüntüyü 'dynamic' stratejisiyle kırpın.
        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

        # 2c. Kırpılmış görüntüye `histogram_equalization` fonksiyonunu uygulayın.
        enhanced_img = histogram_equalization(cropped_img.copy())

        # 2d. Kırpılmış orijinal görüntünün ve iyileştirilmiş görüntünün her bir RGB kanalı için histogramları hesaplayın.
        hist_cropped_r = cv2.calcHist([cropped_img], [0], None, [256], [0, 256])
        hist_cropped_g = cv2.calcHist([cropped_img], [1], None, [256], [0, 256])
        hist_cropped_b = cv2.calcHist([cropped_img], [2], None, [256], [0, 256])

        hist_enhanced_r = cv2.calcHist([enhanced_img], [0], None, [256], [0, 256])
        hist_enhanced_g = cv2.calcHist([enhanced_img], [1], None, [256], [0, 256])
        hist_enhanced_b = cv2.calcHist([enhanced_img], [2], None, [256], [0, 256])

        # 3a. İlk sütunda kırpılmış orijinal RGB görüntüyü gösterin.
        axes[i, 0].imshow(cropped_img)
        axes[i, 0].set_title(f"Orijinal Kırpılmış RGB - Etiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')

        # 3b. İkinci sütunda `histogram_equalization` ile iyileştirilmiş RGB görüntüyü gösterin.
        axes[i, 1].imshow(enhanced_img)
        axes[i, 1].set_title('Histogram Eşitleme ile İyileştirilmiş RGB', fontsize=10)
        axes[i, 1].axis('off')

        # 3c. Üçüncü sütunda kırpılmış orijinal görüntünün RGB histogramlarını gösterin.
        axes[i, 2].plot(hist_cropped_r, color='red')
        axes[i, 2].plot(hist_cropped_g, color='green')
        axes[i, 2].plot(hist_cropped_b, color='blue')
        axes[i, 2].set_title('Orijinal Kırpılmış Histogramlar', fontsize=10)
        axes[i, 2].set_xlim([0, 256])
        axes[i, 2].set_yticks([]) # Remove y-axis ticks for cleaner look

        # 3d. Dördüncü sütunda iyileştirilmiş görüntünün RGB histogramlarını gösterin.
        axes[i, 3].plot(hist_enhanced_r, color='red')
        axes[i, 3].plot(hist_enhanced_g, color='green')
        axes[i, 3].plot(hist_enhanced_b, color='blue')
        axes[i, 3].set_title('İyileştirilmiş Histogramlar', fontsize=10)
        axes[i, 3].set_xlim([0, 256])
        axes[i, 3].set_yticks([]) # Remove y-axis ticks for cleaner look

    except Exception as e:
        print(f"Görüntü işleme veya görselleştirme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(4):
            axes[i, j].axis('off')

# 4. Tüm görselleştirmeleri (`plt.show()`) ile görüntüleyin.
plt.show()

## Gürültü Azaltma Fonksiyonu Tanımlama

In [ ]:
def reduce_noise(img, method='median'):
    """
    Görüntüye belirtilen yönteme göre gürültü azaltma işlemi uygular.

    Args:
        img (numpy.ndarray): Gürültüsü azaltılacak görüntü.
        method (str): Kullanılacak gürültü azaltma yöntemi. 'median' veya 'gaussian' olabilir.

    Returns:
        numpy.ndarray: Gürültüsü azaltılmış görüntü.
    """
    if method == 'median':
        # Median Blur: Tuz-biber gürültüsü için etkilidir. Kernel boyutu tek sayı olmalıdır.
        processed_img = cv2.medianBlur(img, 5) # 5x5 kernel
    elif method == 'gaussian':
        # Gaussian Blur: Normal dağılımlı (Gaussian) gürültüsü için etkilidir.
        processed_img = cv2.GaussianBlur(img, (5, 5), 0) # 5x5 kernel, sigmaX=0 (otomatik hesaplanır)
    else:
        print(f"Hata: Geçersiz gürültü azaltma yöntemi '{method}'. 'median' veya 'gaussian' olmalı.")
        return img # Geçersiz yöntem durumunda orijinal görüntüyü döndür

    return processed_img

print("reduce_noise fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

if 'train_df' not in locals():
    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

# 1. train_df DataFrame'inden rastgele 9 adet görüntü seçin.
num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

# Setup the figure for plotting
fig, axes = plt.subplots(num_images_to_display, 3, figsize=(18, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} g\u00f6r\u00fcnt\u00fc \u00fczerinde g\u00fcr\u00fclt\u00fc azaltma i\u015flemleri g\u00f6rselle\u015ftiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:

        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

        enhanced_img = histogram_equalization(cropped_img.copy())

        denoised_img = reduce_noise(enhanced_img.copy(), method='median')

        axes[i, 0].imshow(cropped_img)
        axes[i, 0].set_title(f"Orijinal K\u0131rp\u0131lm\u0131\u015f RGB - Etiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(enhanced_img)
        axes[i, 1].set_title('Kontrast \u0130yile\u015ftirilmi\u015f RGB', fontsize=10)
        axes[i, 1].axis('off')


        axes[i, 2].imshow(denoised_img)
        axes[i, 2].set_title('G\u00fcr\u00fclt\u00fcs\u00fc Azalt\u0131lm\u0131\u015f RGB (Median Blur)', fontsize=10)
        axes[i, 2].axis('off')

    except Exception as e:
        print(f"G\u00f6r\u00fcnt\u00fc i\u015fleme veya g\u00f6rselle\u015ftirme hatas\u0131: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(3):
            axes[i, j].axis('off')

plt.show()

## Görüntü Segmentasyon Fonksiyonu Tanımlama


In [ ]:
def segment_lesion(img, method='thresholding'):
    """
    Gürültüsü azaltılmış görüntülere belirtilen yöntemle segmentasyon uygular.

    Args:
        img (numpy.ndarray): Segmentasyon uygulanacak görüntü.
        method (str): Kullanılacak segmentasyon yöntemi. 'thresholding' veya 'contour_based' olabilir.

    Returns:
        numpy.ndarray: Segmentasyon maskesi (ikili görüntü).
    """

    # Görüntüyü gri tonlamalıya çevir (eğer zaten gri değilse)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

    if method == 'thresholding':
        # Otsu's eşikleme ile ikili maske oluştur
        _, binary_mask = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary_mask

    elif method == 'contour_based':
        # Görüntüyü hafifçe bulanıklaştır
        blurred_img = cv2.GaussianBlur(gray_img, (5, 5), 0)

        # Canny kenar tespiti
        edges = cv2.Canny(blurred_img, 50, 150) # Alt ve üst eşik değerleri denenebilir

        # Konturları bul
        contours, _ = cv2.findContours(edges.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print("Uyarı: Kontur tabanlı segmentasyon için kontur bulunamadı. Boş maske döndürülüyor.")
            return np.zeros_like(gray_img) # Kontur bulunamazsa boş maske döndür

        # En büyük alanı olan konturu seç
        largest_contour = max(contours, key=cv2.contourArea)

        # En büyük konturu kullanarak maske oluştur
        mask = np.zeros_like(gray_img) # Orijinal görüntü boyutlarında sıfırlardan oluşan maske
        cv2.drawContours(mask, [largest_contour], -1, 255, cv2.FILLED) # Konturu beyaz renk (255) ile doldur
        return mask

    else:
        print(f"Hata: Geçersiz segmentasyon yöntemi '{method}'. 'thresholding' veya 'contour_based' olmalı.")
        return np.zeros_like(gray_img) # Geçersiz yöntem durumunda boş maske döndür

print("segment_lesion fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

if 'train_df' not in locals():
    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

# 1. train_df DataFrame'inden rastgele 9 adet görüntü seçin.
num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 4, figsize=(20, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} görüntü üzerinde segmentasyon işlemleri görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        # 2a. Görüntüyü yükleyin ve RGB formatına dönüştürün.
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 2b. `crop_image` fonksiyonunu kullanarak görüntüyü 'dynamic' stratejisiyle kırpın.
        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

        # 2c. Kırpılmış görüntüye `histogram_equalization` fonksiyonunu uygulayın.
        enhanced_img = histogram_equalization(cropped_img.copy())

        # 2d. İyileştirilmiş görüntüye `reduce_noise` fonksiyonunu uygulayın (Median Blur yöntemiyle).
        denoised_img = reduce_noise(enhanced_img.copy(), method='median')

        # 2e. Gürültüsü azaltılmış görüntüye 'thresholding' segmentasyonunu uygulayın.
        threshold_mask = segment_lesion(denoised_img.copy(), method='thresholding')

        # 2f. Gürültüsü azaltılmış görüntüye 'contour_based' segmentasyonunu uygulayın.
        contour_mask = segment_lesion(denoised_img.copy(), method='contour_based')

        # 3. Her bir rastgele seçilmiş görüntü için dörtlü bir görselleştirme oluşturun:
        # 3a. İlk sütunda gürültüsü azaltılmış orijinal RGB görüntüyü gösterin.
        axes[i, 0].imshow(denoised_img)
        axes[i, 0].set_title(f"Gürültüsü Azaltılmış RGB - Etiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')

        # 3b. İkinci sütunda eşikleme (thresholding) maskesini gösterin.
        axes[i, 1].imshow(threshold_mask, cmap='gray')
        axes[i, 1].set_title('Eşikleme (Thresholding) Maskesi', fontsize=10)
        axes[i, 1].axis('off')

        # 3c. Üçüncü sütunda kontur tabanlı (contour-based) maskesini gösterin.
        axes[i, 2].imshow(contour_mask, cmap='gray')
        axes[i, 2].set_title('Kontur Tabanlı Maske', fontsize=10)
        axes[i, 2].axis('off')

        # 3d. Dördüncü sütunda kontur tabanlı maskeyi orijinal görüntünün üzerinde gösterin.
        # Maskeyi RGB'ye çevir ve yarı şeffaf bindirme yap
        mask_overlay_rgb = cv2.cvtColor(contour_mask, cv2.COLOR_GRAY2BGR)
        alpha = 0.5  # Şeffaflık ayarı
        segmented_overlay = cv2.addWeighted(denoised_img, 1 - alpha, mask_overlay_rgb, alpha, 0)

        axes[i, 3].imshow(segmented_overlay)
        axes[i, 3].set_title('Kontur Maskesi (Overlay)', fontsize=10)
        axes[i, 3].axis('off')

    except Exception as e:
        print(f"Görüntü işleme veya görselleştirme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(4):
            axes[i, j].axis('off')

# 4. Tüm görselleştirmeleri (`plt.show()`) ile görüntüleyin.
plt.show()

In [ ]:
import cv2
import numpy as np
from skimage.filters import try_all_threshold, threshold_yen
from skimage.color import rgb2gray

GLOBAL_THRESHOLD = 127

def segment_lesion(img, method='thresholding'):
    """
    Gürültüsü azaltılmış görüntülere belirtilen yöntemle segmentasyon uygular.

    Args:
        img (numpy.ndarray): Segmentasyon uygulanacak görüntü.
        method (str): Kullanılacak segmentasyon yöntemi. 'thresholding', 'contour_based',
                      'global_thresholding' veya 'skimage_thresholding' olabilir.

    Returns:
        numpy.ndarray: Segmentasyon maskesi (ikili görüntü).
    """

    # Görüntüyü gri tonlamalıya çevir (eğer zaten gri değilse)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

    if method == 'thresholding':
        # Otsu's eşikleme ile ikili maske oluştur
        _, binary_mask = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary_mask

    elif method == 'contour_based':
        # Görüntüyü hafifçe bulanıklaştır
        blurred_img = cv2.GaussianBlur(gray_img, (5, 5), 0)

        # Canny kenar tespiti
        edges = cv2.Canny(blurred_img, 50, 150) # Alt ve üst eşik değerleri denenebilir

        # Konturları bul
        contours, _ = cv2.findContours(edges.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print("Uyarı: Kontur tabanlı segmentasyon için kontur bulunamadı. Boş maske döndürülüyor.")
            return np.zeros_like(gray_img) # Kontur bulunamazsa boş maske döndür

        # En büyük alanı olan konturu seç
        largest_contour = max(contours, key=cv2.contourArea)

        # En büyük konturu kullanarak maske oluştur
        mask = np.zeros_like(gray_img) # Orijinal görüntü boyutlarında sıfırlardan oluşan maske
        cv2.drawContours(mask, [largest_contour], -1, 255, cv2.FILLED) # Konturu beyaz renk (255) ile doldur
        return mask

    elif method == 'global_thresholding':
        # Sabit global eşik değeri ile ikili maske oluştur
        _, binary_mask = cv2.threshold(gray_img, GLOBAL_THRESHOLD, 255, cv2.THRESH_BINARY)
        return binary_mask

    elif method == 'skimage_thresholding':
        threshold_val = threshold_yen(gray_img)
        binary_mask = gray_img > threshold_val
        return (binary_mask * 255).astype(np.uint8)

    else:
        print(f"Hata: Geçersiz segmentasyon yöntemi '{method}'. 'thresholding', 'contour_based', 'global_thresholding' veya 'skimage_thresholding' olmalı.")
        return np.zeros_like(gray_img)

print("segment_lesion fonksiyonu ve gerekli scikit-image kütüphaneleri başarıyla güncellendi.")

In [ ]:
import cv2
import numpy as np
from skimage.filters import threshold_yen
import os

if 'train_df' not in locals():
    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

random_indices = np.random.choice(train_df.index, size=1, replace=False)
sample_idx = random_indices[0]
image_path = train_df.iloc[sample_idx]['image_path']

print(f"Örnek görüntü için eşik değerleri analiz ediliyor: {os.path.basename(image_path)}")

try:
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

    enhanced_img = histogram_equalization(cropped_img.copy())

    denoised_img = reduce_noise(enhanced_img.copy(), method='median')

    gray_denoised_img = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2GRAY)

    global_threshold_val = GLOBAL_THRESHOLD


    _, _ = cv2.threshold(gray_denoised_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu_threshold_val, _ = cv2.threshold(gray_denoised_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    skimage_threshold_val = threshold_yen(gray_denoised_img)

    print(f"\n--- Threshold Values for {os.path.basename(image_path)} ---")
    print(f"Global Threshold (Fixed): {global_threshold_val}")
    print(f"Otsu's Threshold: {otsu_threshold_val:.2f}")
    print(f"Scikit-image (Yen) Threshold: {skimage_threshold_val:.2f}")

except Exception as e:
    print(f"Error processing image {image_path}: {e}")

In [ ]:
import cv2
import numpy as np
from skimage.filters import threshold_yen
import os
import pandas as pd

if 'train_df' not in locals():
    data_path = '/content/Skin cancer ISIC The International Skin Imaging Collaboration/Train'
    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

random_indices = np.random.choice(train_df.index, size=1, replace=False)
sample_idx = random_indices[0]
image_path = train_df.iloc[sample_idx]['image_path']

print(f"Örnek görüntü için eşik değerleri analiz ediliyor: {os.path.basename(image_path)}")

try:
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

    enhanced_img = histogram_equalization(cropped_img.copy())

    denoised_img = reduce_noise(enhanced_img.copy(), method='median')

    gray_denoised_img = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2GRAY)

    global_threshold_val = GLOBAL_THRESHOLD

    otsu_threshold_val, _ = cv2.threshold(gray_denoised_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    skimage_threshold_val = threshold_yen(gray_denoised_img)

    print(f"\n--- Threshold Values for {os.path.basename(image_path)} ---")
    print(f"Global Threshold (Fixed): {global_threshold_val}")
    print(f"Otsu's Threshold: {otsu_threshold_val:.2f}")
    print(f"Scikit-image (Yen) Threshold: {skimage_threshold_val:.2f}")

except Exception as e:
    print(f"Error processing image {image_path}: {e}")

In [ ]:
import cv2
import numpy as np
from skimage.filters import threshold_yen
import os
import pandas as pd
import zipfile

if 'train_df' not in locals():
    # Define paths
    extracted_root_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
    data_path = os.path.join(extracted_root_dir, 'Train')
    zip_file_path = '/content/skin-cancer9-classesisic.zip'
    extract_dir = '/content/'


    if not os.path.exists(data_path):
        print(f"Data directory '{data_path}' not found. Re-downloading and extracting dataset...")
        if not os.path.exists(zip_file_path) or not os.path.exists(extracted_root_dir):
            print("Kaggle dataset zip or extracted folder not found. Running Kaggle download...")
            os.system('kaggle datasets download -d nodoubttome/skin-cancer9-classesisic')
            print("Download complete. Extracting...")

        if os.path.exists(zip_file_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")
        else:
            print(f"Hata: '{zip_file_path}' bulunamadı. Lütfen Kaggle API anahtarlarınızın doğru olduğundan ve verinin indirildiğinden emin olun.")
            raise FileNotFoundError(f"Dataset zip file not found after attempted download: {zip_file_path}")
    else:
        print(f"Data directory '{data_path}' found. Skipping re-download/extraction.")

    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

random_indices = np.random.choice(train_df.index, size=1, replace=False)
sample_idx = random_indices[0]
image_path = train_df.iloc[sample_idx]['image_path']

print(f"Örnek görüntü için eşik değerleri analiz ediliyor: {os.path.basename(image_path)}")

try:

    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Crop the image
    cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

    # Enhance contrast
    enhanced_img = histogram_equalization(cropped_img.copy())

    # Reduce noise (using Median Blur)
    denoised_img = reduce_noise(enhanced_img.copy(), method='median')

    # Convert the denoised image to grayscale for thresholding
    gray_denoised_img = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2GRAY)


    global_threshold_val = GLOBAL_THRESHOLD


    otsu_threshold_val, _ = cv2.threshold(gray_denoised_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    skimage_threshold_val = threshold_yen(gray_denoised_img)

    print(f"\n--- Threshold Values for {os.path.basename(image_path)} ---")
    print(f"Global Threshold (Fixed): {global_threshold_val}")
    print(f"Otsu's Threshold: {otsu_threshold_val:.2f}")
    print(f"Scikit-image (Yen) Threshold: {skimage_threshold_val:.2f}")

except Exception as e:
    print(f"Error processing image {image_path}: {e}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

if 'train_df' not in locals():

    extracted_root_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
    data_path = os.path.join(extracted_root_dir, 'Train')
    zip_file_path = '/content/skin-cancer9-classesisic.zip'
    extract_dir = '/content/'


    if not os.path.exists(data_path):
        print(f"Data directory '{data_path}' not found. Re-downloading and extracting dataset...")

        if not os.path.exists(zip_file_path) or not os.path.exists(extracted_root_dir):
            print("Kaggle dataset zip or extracted folder not found. Running Kaggle download...")
            os.system('kaggle datasets download -d nodoubttome/skin-cancer9-classesisic')
            print("Download complete. Extracting...")

        if os.path.exists(zip_file_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")
        else:
            print(f"Hata: '{zip_file_path}' bulunamadı. Lütfen Kaggle API anahtarlarınızın doğru olduğundan ve verinin indirildiğinden emin olun.")
            raise FileNotFoundError(f"Dataset zip file not found after attempted download: {zip_file_path}")
    else:
        print(f"Data directory '{data_path}' found. Skipping re-download/extraction.")

    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

# 1. train_df DataFrame'inden rastgele 9 adet görüntü seçin.
num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 5, figsize=(25, 5 * num_images_to_display)) # 5 columns for Denoised, Global, Otsu, Skimage, Contour-based
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} görüntü üzerinde segmentasyon işlemleri görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')
        enhanced_img = histogram_equalization(cropped_img.copy())
        denoised_img = reduce_noise(enhanced_img.copy(), method='median')

        # Segmentasyon yöntemlerini uygulama
        global_threshold_mask = segment_lesion(denoised_img.copy(), method='global_thresholding')
        otsu_threshold_mask = segment_lesion(denoised_img.copy(), method='thresholding') # 'thresholding' is Otsu's
        skimage_threshold_mask = segment_lesion(denoised_img.copy(), method='skimage_thresholding')
        contour_mask = segment_lesion(denoised_img.copy(), method='contour_based')

        # --- Görselleştirme ---

        # 1. Sütun: Gürültüsü azaltılmış orijinal RGB görüntü
        axes[i, 0].imshow(denoised_img)
        axes[i, 0].set_title(f"Gürültüsü Azaltılmış RGB\nEtiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')

        # 2. Sütun: Global Eşikleme Maskesi
        axes[i, 1].imshow(global_threshold_mask, cmap='gray')
        axes[i, 1].set_title(f"Global Threshold ({GLOBAL_THRESHOLD})", fontsize=10)
        axes[i, 1].axis('off')

        # 3. Sütun: Otsu Eşikleme Maskesi
        axes[i, 2].imshow(otsu_threshold_mask, cmap='gray')
        axes[i, 2].set_title('Otsu Eşikleme Maskesi', fontsize=10)
        axes[i, 2].axis('off')

        # 4. Sütun: Scikit-image (Yen) Eşikleme Maskesi
        axes[i, 3].imshow(skimage_threshold_mask, cmap='gray')
        axes[i, 3].set_title('Skimage (Yen) Maskesi', fontsize=10)
        axes[i, 3].axis('off')

        # 5. Sütun: Kontur Tabanlı Maske
        axes[i, 4].imshow(contour_mask, cmap='gray')
        axes[i, 4].set_title('Kontur Tabanlı Maske', fontsize=10)
        axes[i, 4].axis('off')


    except Exception as e:
        print(f"Görüntü işleme veya görselleştirme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(5):
            axes[i, j].axis('off')

# 4. Tüm görselleştirmeleri (`plt.show()`) ile görüntüleyin.
plt.show()

In [ ]:
import cv2
import numpy as np

def apply_morphological_operations(binary_mask, kernel):
    """
    İkili maskeye Açma (Opening) ve Kapama (Closing) morfolojik işlemlerini uygular.

    Args:
        binary_mask (numpy.ndarray): Üzerinde işlem yapılacak ikili görüntü maskesi (0 veya 255 değerleri).
        kernel (numpy.ndarray): Morfolojik işlemler için yapısal eleman (kernel).

    Returns:
        numpy.ndarray: Morfolojik işlemlerden geçirilmiş ikili maske.
    """
    # Açma (Opening) işlemi: Erozyon sonra Genişleme. Küçük beyaz gürültüleri kaldırır.
    opened_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # Kapama (Closing) işlemi: Genişleme sonra Erozyon. Küçük siyah delikleri doldurur.
    closed_mask = cv2.morphologyEx(opened_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    return closed_mask

print("apply_morphological_operations fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import pandas as pd

if 'train_df' not in locals():
    extracted_root_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
    data_path = os.path.join(extracted_root_dir, 'Train')
    zip_file_path = '/content/skin-cancer9-classesisic.zip'
    extract_dir = '/content/'

    if not os.path.exists(data_path):
        print(f"Data directory '{data_path}' not found. Re-downloading and extracting dataset...")
        if not os.path.exists(zip_file_path) or not os.path.exists(extracted_root_dir):
            print("Kaggle dataset zip or extracted folder not found. Running Kaggle download...")
            os.system('kaggle datasets download -d nodoubttome/skin-cancer9-classesisic')
            print("Download complete. Extracting...")

        if os.path.exists(zip_file_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")
        else:
            print(f"Hata: '{zip_file_path}' bulunamadı. Lütfen Kaggle API anahtarlarınızın doğru olduğundan ve verinin indirildiğinden emin olun.")
            raise FileNotFoundError(f"Dataset zip file not found after attempted download: {zip_file_path}")
    else:
        print(f"Data directory '{data_path}' found. Skipping re-download/extraction.")

    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 3, figsize=(18, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} görüntü üzerinde morfolojik işlemler görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')
        enhanced_img = histogram_equalization(cropped_img.copy())
        denoised_img = reduce_noise(enhanced_img.copy(), method='median')
        initial_yen_mask = segment_lesion(denoised_img.copy(), method='skimage_thresholding')

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))


        morph_processed_mask = apply_morphological_operations(initial_yen_mask.copy(), kernel)

        axes[i, 0].imshow(denoised_img)
        axes[i, 0].set_title(f"Denoised RGB - Etiket: {row.get('label', 'Yok')}", fontsize=10)
        axes[i, 0].axis('off')


        axes[i, 1].imshow(initial_yen_mask, cmap='gray')
        axes[i, 1].set_title('Initial Yen Mask', fontsize=10)
        axes[i, 1].axis('off')


        axes[i, 2].imshow(morph_processed_mask, cmap='gray')
        axes[i, 2].set_title('Morphologically Processed Mask', fontsize=10)
        axes[i, 2].axis('off')

    except Exception as e:
        print(f"Görüntü işleme veya görselleştirme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(3):
            axes[i, j].axis('off')

plt.show()

 **İkili maske üzerinde Connected Component Labeling (CCL)**

In [ ]:
import cv2
import numpy as np

def perform_ccl(binary_mask):
    """
    İkili maske üzerinde Connected Component Labeling (CCL) uygular
    ve en büyük bağlı bileşeni seçerek yeni bir maske döndürür.

    Args:
        binary_mask (numpy.ndarray): CCL uygulanacak ikili görüntü maskesi (0 veya 255 değerleri).

    Returns:
        numpy.ndarray: Sadece en büyük bağlı bileşeni içeren ikili maske.
    """
    # CCL uygula
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, 8, cv2.CV_32S)

    # Arka planı (etiket 0) hariç tutarak en büyük bileşeni bul
    max_area = 0
    largest_component_label = 0

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area > max_area:
            max_area = area
            largest_component_label = i

    # En büyük bileşeni içeren yeni bir maske oluştur
    largest_component_mask = np.zeros_like(binary_mask, dtype=np.uint8)

    if largest_component_label > 0:
        largest_component_mask[labels == largest_component_label] = 255

    return largest_component_mask

print("perform_ccl fonksiyonu başarıyla tanımlandı.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import pandas as pd

if 'train_df' not in locals():

    extracted_root_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
    data_path = os.path.join(extracted_root_dir, 'Train')
    zip_file_path = '/content/skin-cancer9-classesisic.zip'
    extract_dir = '/content/'


    if not os.path.exists(data_path):
        print(f"Data directory '{data_path}' not found. Re-downloading and extracting dataset...")
        if not os.path.exists(zip_file_path) or not os.path.exists(extracted_root_dir):
            print("Kaggle dataset zip or extracted folder not found. Running Kaggle download...")
            os.system('kaggle datasets download -d nodoubttome/skin-cancer9-classesisic')
            print("Download complete. Extracting...")

        if os.path.exists(zip_file_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")
        else:
            print(f"Hata: '{zip_file_path}' bulunamadı. Lütfen Kaggle API anahtarlarınızın doğru olduğundan ve verinin indirildiğinden emin olun.")
            raise FileNotFoundError(f"Dataset zip file not found after attempted download: {zip_file_path}")
    else:
        print(f"Data directory '{data_path}' found. Skipping re-download/extraction.")

    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

num_images_to_display = 9
random_samples = train_df.sample(n=num_images_to_display, random_state=42)

fig, axes = plt.subplots(num_images_to_display, 4, figsize=(20, 5 * num_images_to_display))
plt.tight_layout(pad=3.0)

print(f"Rastgele seçilen {num_images_to_display} görüntü üzerinde CCL işlemleri görselleştiriliyor...")

for i, (index, row) in enumerate(random_samples.iterrows()):
    image_path = row['image_path']

    try:
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')
        enhanced_img = histogram_equalization(cropped_img.copy())
        denoised_img = reduce_noise(enhanced_img.copy(), method='median')

        initial_yen_mask = segment_lesion(denoised_img.copy(), method='skimage_thresholding')

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

        morph_processed_mask = apply_morphological_operations(initial_yen_mask.copy(), kernel)

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(morph_processed_mask, 8, cv2.CV_32S)

        print(f"Image {os.path.basename(image_path)} - Number of detected components (excluding background): {num_labels - 1}")


        largest_component_mask = perform_ccl(morph_processed_mask.copy())

        # --- Visualization ---
        axes[i, 0].imshow(denoised_img)
        axes[i, 0].set_title(f"Denoised RGB\nEtiket: {row.get('label', 'Yok')}", fontsize=10)

        axes[i, 1].imshow(morph_processed_mask, cmap='gray')
        axes[i, 1].set_title('Morphologically Processed Mask', fontsize=10)


        if num_labels > 1:
            cmap = plt.cm.nipy_spectral
            labeled_img_color = cmap(labels / (num_labels - 1))
            labeled_img_color[labels == 0] = [0, 0, 0, 1]
            labeled_img_display = (labeled_img_color[:, :, :3] * 255).astype(np.uint8)
        else:
            labeled_img_display = np.zeros((*labels.shape, 3), dtype=np.uint8)

        axes[i, 2].imshow(labeled_img_display)
        axes[i, 2].set_title('All Labeled Components (Color-mapped)', fontsize=10)

        axes[i, 3].imshow(largest_component_mask, cmap='gray')
        axes[i, 3].set_title('Largest Connected Component', fontsize=10)

        for j in range(4):
            axes[i, j].axis('off')

    except Exception as e:
        print(f"Görüntü işleme veya görselleştirme hatası: {image_path} - {e}")
        axes[i, 0].set_title(f"Hata: {os.path.basename(image_path)}", fontsize=10)
        for j in range(4):
            axes[i, j].axis('off')


plt.show()

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd

if 'train_df' not in locals():
    extracted_root_dir = '/content/Skin cancer ISIC The International Skin Imaging Collaboration'
    data_path = os.path.join(extracted_root_dir, 'Train')
    zip_file_path = '/content/skin-cancer9-classesisic.zip'
    extract_dir = '/content/'

    if not os.path.exists(data_path):
        print(f"Data directory '{data_path}' not found. Re-downloading and extracting dataset...")
        if not os.path.exists(zip_file_path) or not os.path.exists(extracted_root_dir):
            print("Kaggle dataset zip or extracted folder not found. Running Kaggle download...")
            os.system('kaggle datasets download -d nodoubttome/skin-cancer9-classesisic')
            print("Download complete. Extracting...")

        if os.path.exists(zip_file_path):
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"'{zip_file_path}' dosyası '{extract_dir}' konumuna başarıyla çıkarıldı.")
        else:
            print(f"Hata: '{zip_file_path}' bulunamadı. Lütfen Kaggle API anahtarlarınızın doğru olduğundan ve verinin indirildiğinden emin olun.")
            raise FileNotFoundError(f"Dataset zip file not found after attempted download: {zip_file_path}")
    else:
        print(f"Data directory '{data_path}' found. Skipping re-download/extraction.")

    image_paths = []
    labels = []

    for class_name in os.listdir(data_path):
        class_path = os.path.join(data_path, class_name)

        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    image_paths.append(os.path.join(class_path, img_name))
                    labels.append(class_name)

    data = {'image_path': image_paths, 'label': labels}
    train_df = pd.DataFrame(data)

component_counts = []

print(f"Görüntüler için bileşen dağılımı analiz ediliyor {len(train_df)} görüntü...")

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))

for index, row in train_df.iterrows():
    image_path = row['image_path']

    try:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            print(f"Uyarı: Görüntü yüklenemedi: {image_path}. Atlanıyor.")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        cropped_img = crop_image(img_rgb.copy(), strategy='dynamic')

        enhanced_img = histogram_equalization(cropped_img.copy())

        denoised_img = reduce_noise(enhanced_img.copy(), method='median')

        initial_yen_mask = segment_lesion(denoised_img.copy(), method='skimage_thresholding')

        morph_processed_mask = apply_morphological_operations(initial_yen_mask.copy(), kernel)

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(morph_processed_mask, 8, cv2.CV_32S)

        component_counts.append(num_labels - 1)

    except Exception as e:
        print(f"Hata ({os.path.basename(image_path)}): {e}. Atlanıyor.")
        component_counts.append(0)

if component_counts:
    total_images = len(component_counts)
    one_component_images = sum(1 for count in component_counts if count == 1)
    two_or_more_components_images = sum(1 for count in component_counts if count >= 2)

    min_components = np.min(component_counts)
    max_components = np.max(component_counts)
    avg_components = np.mean(component_counts)

    print("\n--- Connected Component Distribution Summary ---")
    print(f"Total images analyzed: {total_images}")
    print(f"Images with exactly one connected component: {one_component_images} ({one_component_images/total_images:.2%})")
    print(f"Images with two or more connected components: {two_or_more_components_images} ({two_or_more_components_images/total_images:.2%})")
    print(f"Minimum number of components found: {min_components}")
    print(f"Maximum number of components found: {max_components}")
    print(f"Average number of components found: {avg_components:.2f}")
else:
    print("No images were processed for component analysis.")
